# SQLite Database Query - dev.db

This notebook queries the SQLite database to inspect tables, count rows, and display sample data without modifying the database.

## 1. Import Required Libraries

In [ ]:
import sqlite3
import os
from pathlib import Path

print("Libraries imported successfully")

## 2. Connect to the SQLite Database

In [ ]:
db_path = 'dev.db'

# Check if database file exists
if not os.path.exists(db_path):
    print(f"ERROR: Database file '{db_path}' not found!")
    print(f"Current directory: {os.getcwd()}")
    print(f"Files in current directory: {os.listdir('.')}")
else:
    print(f"✅ Database file found at: {os.path.abspath(db_path)}")
    file_size = os.path.getsize(db_path)
    print(f"   File size: {file_size} bytes")

# Connect to database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print("✅ Connected to database successfully")

## 3. Check Table Existence

In [ ]:
# Get all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = cursor.fetchall()

print("\n" + "="*80)
print("TABLES IN DATABASE")
print("="*80)
print(f"Total tables: {len(tables)}")
for table in tables:
    print(f"  - {table[0]}")

target_tables = {'user', 'task', 'conversation', 'message'}
found_tables = {table[0] for table in tables}
print(f"\nTarget tables to check: {sorted(target_tables)}")
print(f"Target tables found: {sorted(target_tables & found_tables)}")
print(f"Target tables MISSING: {sorted(target_tables - found_tables)}")

## 4. Count Rows in Each Table

In [ ]:
# Dictionary to store row counts
row_counts = {}

target_tables = ['user', 'task', 'conversation', 'message']

print("\n" + "="*80)
print("ROW COUNTS")
print("="*80)

for table_name in target_tables:
    try:
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cursor.fetchone()[0]
        row_counts[table_name] = count
        print(f"✅ {table_name:20s}: {count:6d} rows")
    except sqlite3.OperationalError as e:
        print(f"❌ {table_name:20s}: ERROR - {str(e)}")
        row_counts[table_name] = None

## 5. Display Sample Data from Each Table

In [ ]:
print("\n" + "="*80)
print("SAMPLE DATA FROM EACH TABLE (First 5 rows)")
print("="*80)

for table_name in target_tables:
    if row_counts[table_name] is None or row_counts[table_name] == 0:
        print(f"\n[{table_name.upper()}]")
        if row_counts[table_name] is None:
            print("  ❌ Table does not exist")
        else:
            print("  (No data in table)")
        continue
    
    print(f"\n[{table_name.upper()}] - {row_counts[table_name]} total rows")
    
    # Get column info
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = [col[1] for col in cursor.fetchall()]
    print(f"  Columns: {', '.join(columns)}")
    
    # Get first 5 rows
    cursor.execute(f"SELECT * FROM {table_name} LIMIT 5")
    rows = cursor.fetchall()
    
    print(f"  First {len(rows)} rows:")
    for i, row in enumerate(rows, 1):
        print(f"    {i}. {row}")

## 6. Generate Database State Report